# Phase 3: Feature Engineering & Business Metrics
## E-Commerce Sales & Customer Analytics Dashboard

In this notebook, we calculate core business performance KPIs:
1. **Revenue Metrics**: Total Revenue, Average Order Value (AOV), Revenue per Seller/Customer.
2. **Customer Metrics**: Customer Lifetime Value (CLV), Recency-Frequency-Monetary (RFM) Segmentation, Repeat Purchase Rates, and Retention Rates.
3. **Product Metrics**: Revenue Contribution % per product category.
4. **Seller Metrics**: Sales volume, revenue, and ratings.
5. **Delivery Performance**: Delivery durations, shipping lag, and late delivery rates.
6. **Review Metrics**: Aggregated rating distributions.

All calculated metrics will be exported into enriched master files.

---

In [ ]:
import pandas as pd
import numpy as np
import os

CLEANED_DIR = '../data/cleaned/'

# Load cleaned tables
customers = pd.read_csv(os.path.join(CLEANED_DIR, 'customers_cleaned.csv'))
orders = pd.read_csv(os.path.join(CLEANED_DIR, 'orders_cleaned.csv'))
order_items = pd.read_csv(os.path.join(CLEANED_DIR, 'order_items_cleaned.csv'))
order_payments = pd.read_csv(os.path.join(CLEANED_DIR, 'order_payments_cleaned.csv'))
order_reviews = pd.read_csv(os.path.join(CLEANED_DIR, 'order_reviews_cleaned.csv'))
products = pd.read_csv(os.path.join(CLEANED_DIR, 'products_cleaned.csv'))
sellers = pd.read_csv(os.path.join(CLEANED_DIR, 'sellers_cleaned.csv'))

# Convert timestamps
datetime_cols = ['order_purchase_timestamp', 'order_approved_at', 
                 'order_delivered_carrier_date', 'order_delivered_customer_date', 
                 'order_estimated_delivery_date']
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

print('Cleaned tables loaded.')

### 1. Delivery & Fulfillment Metrics
Calculates:
- `delivery_time_days`: Time between purchase and customer receipt.
- `shipping_duration_days`: Time between approval and carrier hand-off.
- `estimated_vs_actual_days`: Promised delivery date vs actual delivery date.
- `is_late_delivery`: Boolean flag indicating if actual delivery exceeded the estimate.

In [ ]:
# Calculate shipping durations in days
orders['delivery_time_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
orders['shipping_duration_days'] = (orders['order_delivered_carrier_date'] - orders['order_approved_at']).dt.total_seconds() / (24 * 3600)
orders['estimated_vs_actual_days'] = (orders['order_estimated_delivery_date'] - orders['order_delivered_customer_date']).dt.total_seconds() / (24 * 3600)

# A positive value for estimated_vs_actual means early delivery, negative means late
orders['is_late_delivery'] = np.where(
    orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date'], 1, 0
)

print(f'Overall late delivery rate: {orders["is_late_delivery"].mean()*100:.2f}%')
print(orders[['delivery_time_days', 'shipping_duration_days', 'estimated_vs_actual_days']].describe())

### 2. Order Revenue Calculation
Calculate total price, total freight and total items per order from `order_items`.

In [ ]:
order_agg = order_items.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum',
    'price_capped': 'sum',
    'freight_capped': 'sum',
    'product_id': 'count'
}).rename(columns={
    'price': 'order_price',
    'freight_value': 'order_freight',
    'price_capped': 'order_price_capped',
    'freight_capped': 'order_freight_capped',
    'product_id': 'order_item_count'
}).reset_index()

order_agg['order_total_value'] = order_agg['order_price'] + order_agg['order_freight']
order_agg['order_total_value_capped'] = order_agg['order_price_capped'] + order_agg['order_freight_capped']

orders_merged = pd.merge(orders, order_agg, on='order_id', how='left')
print(orders_merged[['order_price', 'order_freight', 'order_total_value']].head(3))

### 3. Customer-Level Aggregations & RFM Segmentation
Calculate Recency, Frequency, and Monetary parameters for customers:
- **Recency**: Days since last order relative to the latest order in the dataset.
- **Frequency**: Count of orders placed by customer.
- **Monetary**: Total amount spent by the customer (proxy for Customer Lifetime Value).
- **Repeat Purchase Status**: Identifies customers with more than 1 transaction.

In [ ]:
# Link customer unique ID to order values
cust_orders = pd.merge(orders_merged, customers, on='customer_id', how='left')

# Reference date for recency calculation
latest_date = cust_orders['order_purchase_timestamp'].max()
print(f'Reference latest order date: {latest_date}')

rfm = cust_orders.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (latest_date - x.max()).days,
    'order_id': 'nunique',
    'order_total_value': 'sum'
}).rename(columns={
    'order_purchase_timestamp': 'recency',
    'order_id': 'frequency',
    'order_total_value': 'monetary'
}).reset_index()

# Repeat purchase metric
rfm['is_repeat_buyer'] = np.where(rfm['frequency'] > 1, 1, 0)
repeat_rate = rfm['is_repeat_buyer'].mean()
print(f'Repeat Purchase Rate: {repeat_rate*100:.2f}%')

# Customer Lifetime Value (CLV) = total monetary spent
rfm['clv'] = rfm['monetary']

# Assign RFM scores (1 to 4) using quantiles
rfm['R_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1])  # lower recency is better
rfm['F_score'] = rfm['frequency'].apply(lambda x: 1 if x == 1 else (2 if x == 2 else (3 if x == 3 else 4)))
rfm['M_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4])  # higher monetary is better

# Final RFM Segment categorization
def categorize_rfm(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    score = r + f + m
    if score >= 10:
        return 'Champions'
    elif score >= 8:
        return 'Loyal'
    elif score >= 5:
        return 'Promising/Recent'
    else:
        return 'At Risk/Hibernating'

rfm['customer_segment'] = rfm.apply(categorize_rfm, axis=1)
print(rfm['customer_segment'].value_counts())

# Merge RFM metrics back into customers dataframe
customers_enriched = pd.merge(customers, rfm, on='customer_unique_id', how='left')

### 4. Product Category Performance
Calculate revenue, order counts, and category ranks.

In [ ]:
prod_items = pd.merge(order_items, products, on='product_id', how='left')

category_summary = prod_items.groupby('product_category_name_english').agg({
    'price': 'sum',
    'order_id': 'nunique',
    'product_id': 'count'
}).rename(columns={
    'price': 'total_revenue',
    'order_id': 'total_orders',
    'product_id': 'units_sold'
}).reset_index()

total_rev_all = category_summary['total_revenue'].sum()
category_summary['revenue_contribution_pct'] = (category_summary['total_revenue'] / total_rev_all) * 100
category_summary['category_rank'] = category_summary['total_revenue'].rank(ascending=False, method='min')

display(category_summary.sort_values(by='total_revenue', ascending=False).head(5))

### 5. Seller Performance Metrics
Sellers sales counts, total revenue, and average review scores from order reviews.

In [ ]:
# Link order items with review scores
reviews_orders = pd.merge(order_reviews, order_items, on='order_id', how='inner')

seller_agg = reviews_orders.groupby('seller_id').agg({
    'price': 'sum',
    'review_score': 'mean',
    'order_id': 'nunique'
}).rename(columns={
    'price': 'seller_revenue',
    'review_score': 'seller_avg_rating',
    'order_id': 'seller_order_count'
}).reset_index()

sellers_enriched = pd.merge(sellers, seller_agg, on='seller_id', how='left')
sellers_enriched['seller_revenue'] = sellers_enriched['seller_revenue'].fillna(0)
sellers_enriched['seller_avg_rating'] = sellers_enriched['seller_avg_rating'].fillna(0)

print(sellers_enriched.sort_values(by='seller_revenue', ascending=False).head(3))

### 6. Export Enriched Master Files
We write these enriched master files back to `data/cleaned/` for SQL ingestion and Power BI importing.

In [ ]:
orders_merged.to_csv(os.path.join(CLEANED_DIR, 'orders_master.csv'), index=False)
customers_enriched.to_csv(os.path.join(CLEANED_DIR, 'customers_master.csv'), index=False)
sellers_enriched.to_csv(os.path.join(CLEANED_DIR, 'sellers_master.csv'), index=False)
category_summary.to_csv(os.path.join(CLEANED_DIR, 'category_summary.csv'), index=False)

print('Master analytics files exported successfully!')